For every match, the features are calculated using only matches played
before the current date. The current match result is used only after its
features have been calculated.

Ideas: what would i like to know before beginning a match??
1) how many previous matches each player has played
2) each player's previous win rate 
3) performance in their most recent matches 
4) previous performance on the current surface 
5) ranking difference
6) ranking-points difference 
7) days since the player's previous match 

Even though the previous notebook already sorted the data, each notebook shouldd protect itself against incorrect ordering.

Historical features depend completely on chronological order.

In [2]:
## defaultdict creates dictionaries that automatically provide a default value 
# when a key is encountered for the first time.

##deque stores recent results efficiently.--> retain only the last five match results.

from collections import defaultdict, deque 
from pathlib import Path

import numpy as np
import pandas as pd

In [3]:
PROCESSED_DATA_DIR = Path("../data/processed")

input_file = (
    PROCESSED_DATA_DIR / "matches_neutral_2015_2025.csv"
)

matches = pd.read_csv(
    input_file,
    parse_dates=["Date"]
)

matches = matches.sort_values(
    by=["Date", "MatchID"]
).reset_index(drop=True)

matches["Surface"] = (
    matches["Surface"].fillna("Unknown")
)

print("Dataset shape:", matches.shape)
print("First date:", matches["Date"].min())
print("Last date:", matches["Date"].max())

matches.head()

Dataset shape: (26536, 26)
First date: 2015-01-05 00:00:00
Last date: 2025-11-16 00:00:00


,MatchID,Date,SourceYear,ATP,Location,Tournament,Series,Court,Surface,Round,...,Player2Points,Player1B365Odds,Player2B365Odds,Player1PSOdds,Player2PSOdds,Player1MaxOdds,Player2MaxOdds,Player1AvgOdds,Player2AvgOdds,Player1Won
0,0,2015-01-05,2015,1,Brisbane,Brisbane International,ATP250,Outdoor,Hard,1st Round,...,1195.0,3.50,1.28,3.50,1.34,3.50,1.36,3.30,1.32,0
1,1,2015-01-05,2015,1,Brisbane,Brisbane International,ATP250,Outdoor,Hard,1st Round,...,1730.0,4.50,1.18,4.67,1.23,4.73,1.23,4.31,1.20,1
2,2,2015-01-05,2015,1,Brisbane,Brisbane International,ATP250,Outdoor,Hard,1st Round,...,341.0,1.44,2.62,1.53,2.67,1.53,2.80,1.47,2.62,0
3,3,2015-01-05,2015,1,Brisbane,Brisbane International,ATP250,Outdoor,Hard,1st Round,...,797.0,2.25,1.57,2.37,1.65,2.37,1.67,2.25,1.61,0
4,4,2015-01-05,2015,2,Chennai,Chennai Open,ATP250,Outdoor,Hard,1st Round,...,620.0,1.72,2.00,1.75,2.18,1.80,2.25,1.72,2.07,1


Since a new player may have zero previous matches, division by zero would be impossible. 
Also, after one previous match, an ordinary win rate would be either 0% or 100%. That would be too extreme. 

So i defined a smoothed win rate, in order to give a new player an initial neutral rate of 0.5.

In [4]:
def smoothed_win_rate(wins, matches_played):
    return (wins + 1) / (matches_played + 2)

In [5]:
def recent_win_rate(recent_results ):
    if len(recent_results) == 0:
        return 0.5
    
    return sum(recent_results) / len(recent_results)

##The recent results will contain values such as: [1, 0, 1, 1, 0]
## average = 1+0+1+1+0 / 5 = 0.60

In [6]:
## Historical storage structure 

player_matches = defaultdict(int) ## tot numb of prev matches
player_wins = defaultdict(int) ## tot numb of prev wins

surface_matches = defaultdict(int) ## use pair as dict key
## surface_matches[("Player A", "Clay")] = 20
surface_wins = defaultdict(int) ## victories on a particular surface 

recent_results = defaultdict(
    lambda: deque(maxlen=5)
)

last_match_date = {}

head_to_head_wins = defaultdict(int) ## stores direct victories btw ordered pairs of players 
## head_to_head_wins[("Player A", "Player B")] = 3 
## means player A has previously beaten player B three times.

feature_rows = [] ## to store every transformed match row

1) calculate features using dates before today 
2) add today's results to the history 

### Time-order detail

For a match, I only use results from earlier dates.  
When I reach 2024 or 2025, earlier matches from thosee years can contribute to later matches because thosee results would already be known in a real prediction setting.

Matches on the **same date** are handled together: their features are calculated first, and only after that are their results added to the history. This avoids using another result from the same date by accident.

In [ ]:
for current_date, matches_on_date in matches.groupby(
    "Date", 
    sort=True
):
    for _, row in matches_on_date.iterrows():
        player1 = row["Player1"]
        player2 = row["Player2"]
        surface = row["Surface"]
       
       ## prev matches 
        player1_matches_before = player_matches[player1]
        player2_matches_before = player_matches[player2]
        
        ## prev wins 
        player1_wins_before = player_wins[player1]
        player2_wins_before = player_wins[player2]
        
        ##prev win rates
        player1_win_rate = smoothed_win_rate(
            player1_wins_before, 
            player1_matches_before
        )
        
        player2_win_rate = smoothed_win_rate(
            player2_wins_before, 
            player2_matches_before
        )
        
        ## recent rate 
        player1_recent_rate = recent_win_rate(
            recent_results[player1]
        )
        
        player2_recent_rate = recent_win_rate(
            recent_results[player2]
        )
        
        ## prev matches on current surface 
        player1_surface_matches = surface_matches[
            (player1, surface)
        ]
        
        player2_surface_matches = surface_matches[
            (player2, surface)
        ]
        
        ## prev wins on current surface 
        player1_surface_wins = surface_wins[
            (player1, surface)
        ]
        
        player2_surface_wins = surface_wins[
            (player2, surface)
        ]
        
        ## performance rate based on current surface 
        player1_surface_rate = smoothed_win_rate (
            player1_surface_wins, 
            player1_surface_matches
        )
        
        player2_surface_rate = smoothed_win_rate(
            player2_surface_wins,
            player2_surface_matches
        )
        
        ## prev direct encounters 
        player1_h2h_wins = head_to_head_wins[
            (player1, player2)
        ]
        
        player2_h2h_wins = head_to_head_wins[
            (player2, player1)
        ]
        
        h2h_matches_before = (
            player1_h2h_wins + player2_h2h_wins
        )
        
        ## direct encounters rate 
        player1_h2h_rate = smoothed_win_rate(
            player1_h2h_wins, 
            h2h_matches_before
        )
        
        player2_h2h_rate = smoothed_win_rate(
            player2_h2h_wins, 
            h2h_matches_before
        )
       
       ## time since each player prev match 
        if player1 in last_match_date: 
           player1_days_since_last_match = (
               current_date - last_match_date[player1]
           ).days
        else:
           player1_days_since_last_match = np.nan
           
        if player2 in last_match_date: 
            player2_days_since_last_match = (
                current_date - last_match_date[player2]
            ).days
        else:
            player2_days_since_last_match = np.nan
            
        #missing indicators for days since last match 
        #if not present recieves a value of 1, else 0
        player1_missing_days_since_last_match = int(
            pd.isna(player1_days_since_last_match)
        )
        player2_missing_days_since_last_match = int(
            pd.isna(player2_days_since_last_match)
        )
        
        #difference in days since last match
        if(
            player1_missing_days_since_last_match == 0
            and player2_missing_days_since_last_match == 0
        ):
            days_since_last_match_difference = (
                player1_days_since_last_match - player2_days_since_last_match
            )
        else:
            days_since_last_match_difference = np.nan

        #start from the original neutral match row 
        feature_row = row.to_dict()
        
        #add the newly calculated features 
        feature_row.update(
            {
                "Player1MatchesBefore": player1_matches_before, 
                
                "Player2MatchesBefore": player2_matches_before,
                
                "ExperienceDifference": 
                    player1_matches_before - player2_matches_before, 
                
                "Player1WinRateBefore": player1_win_rate, 
                
                "Player2WinRateBefore": player2_win_rate,
                
                "WinRateDifference": player1_win_rate - player2_win_rate,
                
                "Player1Recent5WinRate": player1_recent_rate, 
                
                "Player2Recent5WinRate": player2_recent_rate, 
                
                "Recent5WinRateDifference": player1_recent_rate - player2_recent_rate, 
                
                "Player1SurfaceMatchesBefore": player1_surface_matches,
                
                "Player2SurfaceMatchesBefore": player2_surface_matches, 
                
                "SurfaceMatchesDifference": player1_surface_matches - player2_surface_matches,
                
                "Player1SurfaceWinRateBefore": player1_surface_rate, 
                
                "Player2SurfaceWinRateBefore": player2_surface_rate, 
                
                "SurfaceWinRateDifference": player1_surface_rate - player2_surface_rate, 
                
                "H2HMatchesBefore": h2h_matches_before, 
                
                "Player1H2HWinRateBefore": player1_h2h_rate, 
                
                "Player2H2HWinRateBefore": player2_h2h_rate,
                
                "DifferenceH2HWinRateBefore": player1_h2h_rate - player2_h2h_rate, 
                
                "Player1DaysSinceLastMatch": player1_days_since_last_match,
                
                "Player2DaysSinceLastMatch": player2_days_since_last_match, 
                
                "DaysSinceLastMatchDifference": days_since_last_match_difference,
                
                "MissingPlayer1DaysSinceLastMatch": player1_missing_days_since_last_match,
                
                "MissingPlayer2DaysSinceLastMatch": player2_missing_days_since_last_match,
                
                "RankDifference": row["Player2Rank"] - row["Player1Rank"], 
                
                "PointsDifference": row["Player1Points"] - row["Player2Points"]
            }
        )
        feature_rows.append(feature_row)
    
    for _, row in matches_on_date.iterrows(): 
        
        player1= row["Player1"]
        player2 = row["Player2"]
        surface = row["Surface"]
        
        if row["Player1Won"] == 1: 
            winner = player1
            loser = player2
            
            player1_result = 1
            player2_result = 0
        else: 
            winner = player2
            loser = player1
            
            player1_result = 0
            player2_result = 1
            
        ## update match count 
        player_matches[player1] +=1
        player_matches[player2] +=1
        
        ##update win counts
        player_wins[winner] +=1
        
        ##update surface match counts 
        surface_matches[(player1, surface)] +=1
        surface_matches[(player2, surface)] +=1
        
        ##update surface win counts
        surface_wins[(winner, surface)] +=1
        
        ##update recent results 
        recent_results[player1].append(player1_result)
        recent_results[player2].append(player2_result)
        
        ##update latest match dates 
        last_match_date[player1] = current_date 
        last_match_date[player2] = current_date
        
        ## update direct encounters
        head_to_head_wins[(winner, loser)] +=1
        
        

In [8]:
featured_matches = pd.DataFrame(feature_rows)

print("Original shape:", matches.shape)
print("Featured shape:", featured_matches.shape)

Original shape: (26536, 26)
Featured shape: (26536, 52)


inspect the new features --> for the earliest dates the new features added shouldd have neutral values. 
ex: MatchesBefore = 0, 
    WinRateBefore = 0.5
    Recent5WinRate = 0.5
    SurfaceWinRate = 0.5
    H2HWinRate = 0.5, etc.

In [9]:
feature_columns = [
    "Player1MatchesBefore", 
    "Player2MatchesBefore",
    "ExperienceDifference",
    
    "Player1WinRateBefore",
    "Player2WinRateBefore",
    "WinRateDifference",
    
    "Player1Recent5WinRate",
    "Player2Recent5WinRate",
    "Recent5WinRateDifference",
    
    "Player1SurfaceMatchesBefore",
    "Player2SurfaceMatchesBefore",
    "SurfaceMatchesDifference",
    
    "Player1SurfaceWinRateBefore",
    "Player2SurfaceWinRateBefore",
    "SurfaceWinRateDifference",
    
    "H2HMatchesBefore",
    
    "Player1H2HWinRateBefore",
    "Player2H2HWinRateBefore",
    "DifferenceH2HWinRateBefore",
    
    "Player1DaysSinceLastMatch",
    "Player2DaysSinceLastMatch",
    "DaysSinceLastMatchDifference",
    
    "MissingPlayer1DaysSinceLastMatch",
    "MissingPlayer2DaysSinceLastMatch",
    
    "RankDifference",
    "PointsDifference"
]

featured_matches[
    [
        "Date",
        "Player1",
        "Player2",
        "Surface",
        "Player1Won"
    ] + feature_columns
].head(10)

,Date,Player1,Player2,Surface,Player1Won,Player1MatchesBefore,Player2MatchesBefore,ExperienceDifference,Player1WinRateBefore,Player2WinRateBefore,...,Player1H2HWinRateBefore,Player2H2HWinRateBefore,DifferenceH2HWinRateBefore,Player1DaysSinceLastMatch,Player2DaysSinceLastMatch,DaysSinceLastMatchDifference,MissingPlayer1DaysSinceLastMatch,MissingPlayer2DaysSinceLastMatch,RankDifference,PointsDifference
0,2015-01-05,Golubev A.,Chardy J.,Hard,0,0,0,0,0.5,0.5,...,0.5,0.5,0.0,NaN,NaN,NaN,1,1,-41.0,-504.0
1,2015-01-05,Duckworth J.,Simon G.,Hard,1,0,0,0,0.5,0.5,...,0.5,0.5,0.0,NaN,NaN,NaN,1,1,-104.0,-1300.0
2,2015-01-05,Benneteau J.,Kokkinakis T.,Hard,0,0,0,0,0.5,0.5,...,0.5,0.5,0.0,NaN,NaN,NaN,1,1,124.0,1024.0
3,2015-01-05,Querrey S.,Tomic B.,Hard,0,0,0,0,0.5,0.5,...,0.5,0.5,0.0,NaN,NaN,NaN,1,1,18.0,293.0
4,2015-01-05,Coric B.,Haase R.,Hard,1,0,0,0,0.5,0.5,...,0.5,0.5,0.0,NaN,NaN,NaN,1,1,-15.0,-63.0
5,2015-01-05,Roger-Vasselin E.,Muller G.,Hard,0,0,0,0,0.5,0.5,...,0.5,0.5,0.0,NaN,NaN,NaN,1,1,-70.0,-385.0
6,2015-01-05,Becker B.,Bolelli S.,Hard,0,0,0,0,0.5,0.5,...,0.5,0.5,0.0,NaN,NaN,NaN,1,1,12.0,163.0
7,2015-01-05,Lorenzi P.,Brown D.,Hard,0,0,0,0,0.5,0.5,...,0.5,0.5,0.0,NaN,NaN,NaN,1,1,35.0,210.0
8,2015-01-05,Dodig I.,Safwat M.,Hard,1,0,0,0,0.5,0.5,...,0.5,0.5,0.0,NaN,NaN,NaN,1,1,200.0,407.0
9,2015-01-05,Gasquet R.,Andujar P.,Hard,1,0,0,0,0.5,0.5,...,0.5,0.5,0.0,NaN,NaN,NaN,1,1,15.0,400.0


Note:: 
1) RANK DIFFERENCE = (rank of player2) - (rank of player1) 
    smaller ranking in tennis is better !!
    so... 
    1) if RankDifference is positive --> Player1 has better ranking 
    2) else RankDifference is negative --> Player2 has better ranking
2) POINTS DIFFERENCE  = player1Points - player2Points
    --> positive if player1 has more points. 


validation check for the new featured_matches..
1) Player1Won must always have value 0 or 1
2) the match IDs have to be different
3) every Rate calculation must be btw 0 and 1
4) number of rows in the original dataset == number of rows in the dataset were the features were added


In [10]:
assert len(featured_matches) == len(matches)

assert featured_matches["MatchID"].is_unique

assert featured_matches["Player1Won"].isin([0, 1]).all()

assert featured_matches["Player1WinRateBefore"].between(0, 1).all()
assert featured_matches["Player2WinRateBefore"].between(0, 1).all()

assert featured_matches["Player1SurfaceWinRateBefore"].between(0, 1).all()
assert featured_matches["Player2SurfaceWinRateBefore"].between(0, 1).all()

assert featured_matches["Player1H2HWinRateBefore"].between(0, 1).all()
assert featured_matches["Player2H2HWinRateBefore"].between(0, 1).all()

#missing value for days is binary 
assert featured_matches["MissingPlayer1DaysSinceLastMatch"].isin([0, 1]).all()
assert featured_matches["MissingPlayer2DaysSinceLastMatch"].isin([0, 1]).all()

print("All validation check completed.")

All validation check completed.


In [11]:
## check for missing values 
featured_matches[feature_columns].isna().sum().sort_values(ascending=False)

DaysSinceLastMatchDifference        26536
Player2DaysSinceLastMatch             412
Player1DaysSinceLastMatch             392
ExperienceDifference                    0
Player2WinRateBefore                    0
WinRateDifference                       0
Player1Recent5WinRate                   0
Player1WinRateBefore                    0
Player1MatchesBefore                    0
Player2MatchesBefore                    0
Player1SurfaceMatchesBefore             0
Recent5WinRateDifference                0
Player2Recent5WinRate                   0
Player2SurfaceMatchesBefore             0
SurfaceWinRateDifference                0
SurfaceMatchesDifference                0
Player1SurfaceWinRateBefore             0
Player2SurfaceWinRateBefore             0
Player2H2HWinRateBefore                 0
Player1H2HWinRateBefore                 0
H2HMatchesBefore                        0
DifferenceH2HWinRateBefore              0
MissingPlayer1DaysSinceLastMatch        0
MissingPlayer2DaysSinceLastMatch  

Missing values in `Player1DaysSinceLastMatch`,
`Player2DaysSinceLastMatch` and `RestDaysDifference` are normal for players who are appearing for the first time in the available dataset.
For the two original rest-day features, I also created binary missing indicators. Numerical missing values will later be imputed using values calculated only from the training set, in order to avoid data leakage.

In [12]:
first_date = featured_matches["Date"].min()

featured_matches.loc[
    featured_matches["Date"].eq(first_date), 
    [
        "Date",
        "Player1",
        "Player2",
        "Player1MatchesBefore", #shouldd be zero
        "Player2MatchesBefore", #shouldd be zero
        "H2HMatchesBefore" #shouldd be zero
    ]
].head(20)

,Date,Player1,Player2,Player1MatchesBefore,Player2MatchesBefore,H2HMatchesBefore
0,2015-01-05,Golubev A.,Chardy J.,0,0,0
1,2015-01-05,Duckworth J.,Simon G.,0,0,0
2,2015-01-05,Benneteau J.,Kokkinakis T.,0,0,0
3,2015-01-05,Querrey S.,Tomic B.,0,0,0
4,2015-01-05,Coric B.,Haase R.,0,0,0
5,2015-01-05,Roger-Vasselin E.,Muller G.,0,0,0
6,2015-01-05,Becker B.,Bolelli S.,0,0,0
7,2015-01-05,Lorenzi P.,Brown D.,0,0,0
8,2015-01-05,Dodig I.,Safwat M.,0,0,0
9,2015-01-05,Gasquet R.,Andujar P.,0,0,0


In [13]:
# verify that history increases over time

all_players = pd.concat(
    [
        featured_matches["Player1"],
        featured_matches["Player2"]
    ],
    ignore_index=True
)

example_player = all_players.value_counts().index[0]

print("Example player:", example_player)

Example player: Zverev A.


In [14]:
example_player_matches = featured_matches.loc[
    featured_matches["Player1"].eq(example_player)
    | featured_matches["Player2"].eq(example_player),
    [
        "Date",
        "Player1",
        "Player2",
        "Player1MatchesBefore",
        "Player2MatchesBefore",
        "Player1WinRateBefore",
        "Player2WinRateBefore",
        "Player1Won" 
    ]
].sort_values("Date")

example_player_matches.head(15)

,Date,Player1,Player2,Player1MatchesBefore,Player2MatchesBefore,Player1WinRateBefore,Player2WinRateBefore,Player1Won
341,2015-02-09,Zverev A.,Bautista R.,0,5,0.500000,0.571429,0
441,2015-02-17,Monfils G.,Zverev A.,8,1,0.600000,0.333333,1
533,2015-02-24,Zverev A.,Ilhan M.,2,2,0.250000,0.250000,0
704,2015-03-26,Zverev A.,Groth S.,3,12,0.200000,0.428571,1
730,2015-03-28,Rosol L.,Zverev A.,9,4,0.363636,0.333333,1
947,2015-04-27,Becker B.,Zverev A.,11,5,0.307692,0.285714,0
985,2015-04-29,Zverev A.,Kohlschreiber P.,6,15,0.375000,0.470588,0
1318,2015-06-09,Pavic M.,Zverev A.,0,7,0.500000,0.333333,0
1327,2015-06-10,Zverev A.,Troicki V.,8,30,0.400000,0.593750,0
1354,2015-06-15,Nieminen J.,Zverev A.,20,9,0.409091,0.363636,0


In [15]:
## save the featured dataset

output_file = (
    PROCESSED_DATA_DIR / "featured_matches_2015_2025.csv"
)

featured_matches.to_csv(
    output_file, 
    index = False
)

print("Featured dataset saved to:")
print(output_file.resolve())

Featured dataset saved to:
C:\Users\User\Downloads\project_ai\data\processed\featured_matches_2015_2025.csv


In [16]:
## read the file back 

saved_featured_matches = pd.read_csv(
    output_file, 
    parse_dates=["Date"]
)

print("Saved featured shape:", saved_featured_matches.shape)

saved_featured_matches[
    [
        "Date",
        "Player1",
        "Player2",
        "RankDifference",
        "WinRateDifference",
        "SurfaceWinRateDifference",
        "Player1Won" 
    ]
].head()

Saved featured shape: (26536, 52)


,Date,Player1,Player2,RankDifference,WinRateDifference,SurfaceWinRateDifference,Player1Won
0,2015-01-05,Golubev A.,Chardy J.,-41.0,0.0,0.0,0
1,2015-01-05,Duckworth J.,Simon G.,-104.0,0.0,0.0,1
2,2015-01-05,Benneteau J.,Kokkinakis T.,124.0,0.0,0.0,0
3,2015-01-05,Querrey S.,Tomic B.,18.0,0.0,0.0,0
4,2015-01-05,Coric B.,Haase R.,-15.0,0.0,0.0,1


In [17]:
## final feature summary

print("Feature shape:", featured_matches.shape)

print(featured_matches[feature_columns].isna().sum())

print(
    featured_matches[
        [
            "Player1Won",
            "RankDifference",
            "WinRateDifference",
            "Recent5WinRateDifference",
            "SurfaceMatchesDifference",
            "SurfaceWinRateDifference",
            "H2HMatchesBefore",
            "DaysSinceLastMatchDifference"
        ]
    ].describe()
)

Feature shape: (26536, 52)
Player1MatchesBefore                    0
Player2MatchesBefore                    0
ExperienceDifference                    0
Player1WinRateBefore                    0
Player2WinRateBefore                    0
WinRateDifference                       0
Player1Recent5WinRate                   0
Player2Recent5WinRate                   0
Recent5WinRateDifference                0
Player1SurfaceMatchesBefore             0
Player2SurfaceMatchesBefore             0
SurfaceMatchesDifference                0
Player1SurfaceWinRateBefore             0
Player2SurfaceWinRateBefore             0
SurfaceWinRateDifference                0
H2HMatchesBefore                        0
Player1H2HWinRateBefore                 0
Player2H2HWinRateBefore                 0
DifferenceH2HWinRateBefore              0
Player1DaysSinceLastMatch             392
Player2DaysSinceLastMatch             412
DaysSinceLastMatchDifference        26536
MissingPlayer1DaysSinceLastMatch        0
Missing